In [0]:
%run ./00_config

In [0]:
%run ./01_odoo_client

In [0]:
%run ./02_helpers

In [0]:
%run ./03_transform_load

In [0]:
import json
from datetime import datetime, timezone

client = OdooClient(ODOO_URL, ODOO_DB, ODOO_USER, ODOO_PASSWORD)

def sanitize_record(record):
    sanitized = {}
    for key, value in record.items():
        if isinstance(value, (dict, list)):
            sanitized[key] = json.dumps(value) if value else None
        elif value is False:
            sanitized[key] = None
        else:
            sanitized[key] = value
    return sanitized

def run_pipeline(model):
    try:
        last_write = get_last_write(model)

        records = client.call(
            model,
            "search_read",
            args=[[["write_date", ">", last_write]]],
            kwargs={"limit": 10000}
        )

        # Sanitize before normalize to fix type inference errors
        records = [sanitize_record(r) for r in records]
        records = normalize(records)

        if records:
            load_to_bronze(records, model)
            update_tracking(model, records)

        print(f"✅ Done: {model}")

    except Exception as e:
        print(f"❌ Error: {model} -> {e}")

In [0]:
DEPARTMENTS = [
    # Partner

    "res.partner",
    "res.partner.category",
    "res.groups",
    "res.users",
    # Product

    "product.product",
    "product.template",
    "product.category",
    "product.combo",
    "product.combo.item",
    "product.pricelist",
    "product.pricelist.item",
    "product.tag",
    #Invoice

    "account.account",
    "account.analytic.account",
    "account.analytic.line",
    "account.bank.statement",
    "account.bank.statement.line",
    "account.move",
    "account.move.line",
    "account.journal",
    "account.tax",
    "account.tax.group",
    "account.tax.repartition.line",
    "account.payment",
    "account.payment.term",
    # Sales

    "sale.order",
    "sale.order.line",
    "sale.order.template",
    # Purchase

    "purchase.order",
    "purchase.order.line",
    # Stock

    "stock.move",
    "stock.move.line",
    "stock.picking",
    "stock.quant",
    "stock.location",
    "stock.warehouse",
    # Crm

    "crm.lead",
    "crm.stage",
    "crm.team",
    "utm.campaign",
    "utm.source",
    "utm.medium",
    # Hr 

    "hr.employee",
    "hr.department",
    "hr.job"
]

In [0]:
for model in DEPARTMENTS:
    run_pipeline(model)